# Week 3: Passwords, Cracking, and Defense
**Goal:** test a leaked password database against a wordlist, see how fast weak passwords fall, then build the defenses that stop it.

**Time:** about 2 hours. **You need:** Weeks 1 and 2 (files, dictionaries, `hashlib`).

**Story.** A fictional game company, Riverside Games, was breached. The attacker posted 20 usernames with
**unsalted SHA-256** password hashes. Your team must answer: *how many accounts are at risk, and how should
passwords be stored and chosen from now on?*

> Every username and password here is invented for this lab. Testing real people's passwords without permission is
> illegal. Only work with the hashes in this notebook.

## Part 0: Get the data (run this cell)

In [ ]:
import os, urllib.request, csv, hashlib, time

DATA_URL = "https://raw.githubusercontent.com/Pantea1358/Cyber-labs-/main/weeks/week03-passwords/data/"
os.makedirs("data", exist_ok=True)
for name in ["riverside_games_leak.csv", "common_passwords.txt"]:
    if not os.path.exists(f"data/{name}"):
        urllib.request.urlretrieve(DATA_URL + name, f"data/{name}")

def sha256(text):
    return hashlib.sha256(text.encode()).hexdigest()

print("Data ready:", os.listdir("data"))

## Part 1: Load the leak and the wordlist (10 min)
1. Read `data/riverside_games_leak.csv` into a dictionary `leak` that maps **username to hash**.
2. Read `data/common_passwords.txt` into a list `wordlist`.
3. Print how many users and how many words you have.

*Hint:* `csv.DictReader(f)` gives each row as a dictionary with keys `username` and `sha256`.

In [ ]:
leak = {}      # TODO: username -> hash
wordlist = []  # TODO: list of common passwords

print(len(leak), "users,", len(wordlist), "words")

## Part 2: Matching hashes against a wordlist (20 min)
A defender who steals a hash database can test a list of common passwords: hash each word and see if it matches
any stored hash. Because SHA-256 is fast and these hashes have no salt, this is quick.

First build a lookup the smart way. Hash every word in the wordlist **once** into a dictionary
`hash_to_word` that maps hash to the word. Then a user is recovered if their stored hash is a key in it.

Fill `recovered` (username to password) and print each one. How many of the 20 fell?

In [ ]:
# TODO: hash every word once into hash_to_word = {hash: word}
hash_to_word = {}

recovered = {}
# TODO: for each user, if their hash is in hash_to_word, store the word in `recovered`

for user, pw in recovered.items():
    print(f"{user:<10} {pw}")
print(f"\n{len(recovered)} of {len(leak)} accounts recovered")

**Question 1:** why is it much faster to hash the wordlist once into a dictionary than to re-hash every word
for every user? Think about how many hashes each approach computes.

*Your answer:*

## Part 3: Mangling rules, how attackers get more (20 min)
Some users were not recovered because they added a capital letter, a number, or a year. Attackers apply
**rules** to each word to cover these tricks. Write a function `variants(word)` that yields the word plus
common variations, then try all of them against the still-uncracked users.

Cover at least: the word itself, capitalized, with `123`, with `!`, with the years `2024`/`2025`/`2026`,
and capitalized + a year + `!` (for example `Summer2026!`).

In [ ]:
def variants(word):
    # TODO: return a set with the word and its common variations
    return {word}

still_locked = {u: h for u, h in leak.items() if u not in recovered}
# TODO: for each still-locked user, try every variant of every word; store any match in `recovered`

print(f"\n{len(recovered)} of {len(leak)} accounts recovered after rules")

**Question 2:** list the users you still could not recover. Look at the style of the passwords you *did*
recover versus the ones you could not. What makes the survivors hard to guess?

*Your answer:*

## Part 4: A password strength meter (20 min)
Now switch sides. Write `strength(password)` that returns `"weak"`, `"medium"`, or `"strong"` based on simple rules,
and explains why. A reasonable scheme:

- **weak:** shorter than 12 characters, OR the lowercase form (stripped of trailing digits/`!`) is in the wordlist.
- **strong:** 16 or more characters, OR four or more words separated by spaces or dashes (a passphrase).
- **medium:** everything else.

Test it on a few of the recovered passwords and a few survivors.

In [ ]:
wordset = set(wordlist)

def strength(password):
    # TODO: return "weak", "medium", or "strong" using the rules above
    return "medium"

for pw in ["dragon123", "Summer2026!", "P@ssw0rd", "correct-horse-violin-sunset", "Gx7#qL9!vR2@pZ"]:
    print(f"{strength(pw):<8} {pw}")

## Part 5: The real fix, slow salted hashing (20 min)
The breach was so damaging because the hashes were **fast and unsalted**. The fix is a **slow, salted** hash
like bcrypt. Salt means two users with the same password get different hashes, so one wordlist run cannot crack
everyone at once. Slow means each guess costs real time.

Run this to feel the difference. It hashes one password with SHA-256 and with bcrypt and times each.

In [ ]:
import bcrypt

password = b"dragon123"

start = time.perf_counter()
for _ in range(10000):
    hashlib.sha256(password).hexdigest()
sha_time = time.perf_counter() - start

start = time.perf_counter()
bcrypt.hashpw(password, bcrypt.gensalt(rounds=12))
bcrypt_time = time.perf_counter() - start

print(f"10,000 SHA-256 hashes: {sha_time:.4f} s")
print(f"1 bcrypt hash:         {bcrypt_time:.4f} s")
print(f"bcrypt is about {bcrypt_time / (sha_time / 10000):,.0f}x slower per guess")

Now store two users' passwords with bcrypt and verify a login the right way. Notice the two hashes differ
even when the password is the same, because of the salt.

In [ ]:
def store(password):
    # TODO: return a bcrypt hash of the password (use bcrypt.gensalt(rounds=12))
    return b""

def verify(password, stored):
    # TODO: return True if the password matches the stored bcrypt hash
    return False

h1 = store("dragon123")
h2 = store("dragon123")
print("Same password, two hashes:")
print(" ", h1.decode())
print(" ", h2.decode())
print("Both verify:", verify("dragon123", h1), verify("dragon123", h2))
print("Wrong password:", verify("dragon124", h1))

**Question 3:** your Part 2 attack recovered many accounts in well under a second. Using the "x slower"
number above, roughly how long would the *same* wordlist run take against bcrypt hashes, for all 20 users?
Why does salting also break the one-dictionary-cracks-everyone shortcut?

*Your answer:*

## Hands-on challenge: the breach report (15 min)
You are presenting to Riverside Games. Build a report, as a list of lines, that includes:

1. How many of the 20 accounts you recovered, as a count and a percentage.
2. The three most common patterns you saw (for example "word + year + !").
3. One sentence on how passwords should be *stored* from now on.
4. One sentence on what to require when users *choose* a password, using your `strength` meter.

Print it. This is the kind of summary a security analyst actually writes.

In [ ]:
# TODO: build the report as a list of strings, then print it joined by newlines
report = []
print("\n".join(report))

## Part 7: Reflect (10 min)
**Exit ticket.** Answer in 2 to 3 sentences each:

1. You now know how to run a dictionary attack. Name a situation where doing this is legal and useful, and one where
   it would be a crime.
2. A friend says "I'll just add `!` to the end of my password to make it safe." Based on today, what do you tell them?

*Your answers:*

**Stretch (optional):** add a rule to `variants()` for leetspeak (`a` to `@`, `o` to `0`, `i` to `1`, `s` to `$`,
`e` to `3`). Does it recover any more accounts?